### 1. 패키지 설치

### 2. 문서 split 및 Chroma를 활용한 vector store 구성

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200 # k=100이면 제대로 안나옴
)

loader = Docx2txtLoader('./dataset_part_1.docx')
document_list = loader.load_and_split(text_splitter)

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings

# 올바른 Hugging Face 모델을 사용한 임베딩 생성
embeddings = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large')

# 확인
print("HuggingFaceEmbeddings initialized successfully!")


/tmp/ipykernel_1956/746791366.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large')
/home/chae/ch/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


HuggingFaceEmbeddings initialized successfully!


In [3]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

# 3. Chroma 데이터베이스 초기화 및 문서 추가
collection_name = 'chroma_art'

database = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory='./chroma_huggingface'
)

print("Chroma database initialized successfully!")

Chroma database initialized successfully!


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

In [5]:
import torch
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# 모델과 토크나이저 로드 (CUDA 사용)
model_id = 'LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="cuda"  # CUDA에서 자동 배치
)


Loading checkpoint shards: 100%|██████████| 7/7 [00:23<00:00,  3.42s/it]


In [7]:
# 파이프라인 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,  # 생성할 최대 토큰 수 증가
    do_sample=True,        # 샘플링 활성화
    temperature=0.9,      # 다양성 증가
    top_k=50,             # 상위 k개 토큰 중에서 샘플링
    repetition_penalty=1.03
)
# LangChain의 HuggingFacePipeline 사용
llm = HuggingFacePipeline(pipeline=pipe)

In [9]:
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.output_parsers.string import StrOutputParser # module path 수정
from langchain.prompts import ChatPromptTemplate

template = '''
친절한 챗봇으로서 질문에 거짓 없이 대답해줘. 
모든 대답은 한국어(Korean)으로 대답해줘.
2-3줄 정도로 간단하게 대답해줘.

{context}

Question: {question}
'''
prompt = ChatPromptTemplate.from_template(template)

In [10]:
retriever = database.as_retriever(search_kwargs={"k": 2})

In [11]:
from langchain.schema.runnable import RunnablePassthrough, RunnableMap

# 체인 구성 수정
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [16]:
query = "김기승의 낙양성동도이화의 작품번호는 무엇이죠?"

In [17]:
# 마지막에 원하는 형식으로 출력 필터링
response = chain.invoke(query)
print(response)

Human: 
친절한 챗봇으로서 질문에 거짓 없이 대답해줘. 
모든 대답은 한국어(Korean)으로 대답해줘.
2-3줄 정도로 간단하게 대답해줘.

[Document(metadata={'source': './dataset_part_1.docx'}, page_content='작품명: 낙양성동도이화(6곡병) / 洛陽城東桃李花(六曲屛) / \n작품 번호: 01370\n\n\n작가: 김기승 / KIM Kiseung\n\n제작 연도: 1963\n크기: 128×31.5×(6)\n재료: 종이에 먹; 6폭 병풍\n카테고리: 서예\n\n작품 설명'), Document(metadata={'source': './dataset_part_1.docx'}, page_content='작품명: 그가 열방사이에- 이사야(2곡병) / N/A / \n작품 번호: 01335\n\n\n작가: 김기승 / KIM Kiseung\n\n제작 연도: 1972\n크기: 230×60×(2)\n재료: 종이에 먹; 가리개\n카테고리: 서예\n\n작품 설명')]

Question: 김기승의 낙양성동도이화의 작품번호는 무엇이죠?

Answer: 김기승의 "낙양성동도이화(6곡병)"의 작품번호는 01370입니다. 이 작품은 1963년에 제작되었으며, 크기는 128×31.5×6이고 재료는 종이에 먹을 사용하여 만든 6폭 병풍입니다. 카태고리는 서예입니다.


In [18]:
import re

def extract_answer(text):
    # 정규 표현식을 사용하여 'Answer:' 이후의 모든 텍스트 추출
    match = re.search(r"Answer:\s*(.*)", text, re.DOTALL)
    if match:
        answer = match.group(1).strip()  # 불필요한 공백 제거
    else:
        answer = "답변을 찾을 수 없습니다."
    
    return answer

In [19]:
answer = extract_answer(response)
print(answer)

김기승의 "낙양성동도이화(6곡병)"의 작품번호는 01370입니다. 이 작품은 1963년에 제작되었으며, 크기는 128×31.5×6이고 재료는 종이에 먹을 사용하여 만든 6폭 병풍입니다. 카태고리는 서예입니다.
